# Task 5: Model Evaluation and Kaggle Submission

In this notebook, I load the best-performing EfficientNet-B0 checkpoint and use it to predict the classes of the unseen test images. The predictions are then added to a copy of the official sample submission file so that the image IDs, row order, column names, and class names remain in the exact format required by Kaggle.

In [11]:
from pathlib import Path

import pandas as pd
import torch
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

TEST_DIR = PROJECT_ROOT / "data" / "raw" / "test" / "test"
SAMPLE_SUBMISSION_PATH = (
    PROJECT_ROOT / "data" / "raw" / "sample_submission.csv"
)
BEST_MODEL_PATH = (
    PROJECT_ROOT / "models" / "efficientnet_b0_best.pth"
)
SUBMISSIONS_DIR = PROJECT_ROOT / "submissions"
SUBMISSIONS_DIR.mkdir(parents=True, exist_ok=True)

sample_submission = pd.read_csv(
    SAMPLE_SUBMISSION_PATH,
    dtype={"image_id": str}
)

if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("Test directory exists:", TEST_DIR.exists())
print("Best model exists:", BEST_MODEL_PATH.exists())
print("Submission rows:", len(sample_submission))
print("Device:", device)

display(sample_submission.head())

Test directory exists: True
Best model exists: True
Submission rows: 29443
Device: mps


,image_id,class
0,1,DogEye
1,2,CatNose
2,3,CatNose
3,4,DogFace
4,5,DogMouth


In [12]:
IMAGE_SIZE = 224
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

test_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
])


class TestDataset(Dataset):
    def __init__(self, dataframe, image_directory, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.image_directory = image_directory
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        image_id = self.dataframe.iloc[index]["image_id"]

        # Convert submission ID 1 into filename 00001.jpg
        image_filename = f"{str(image_id).zfill(5)}.jpg"
        image_path = self.image_directory / image_filename

        image = Image.open(image_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image


test_dataset = TestDataset(
    sample_submission,
    TEST_DIR,
    transform=test_transform
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0
)

test_batch = next(iter(test_loader))

print("Test dataset size:", len(test_dataset))
print("Number of batches:", len(test_loader))
print("Test batch shape:", test_batch.shape)

Test dataset size: 29443
Number of batches: 921
Test batch shape: torch.Size([32, 3, 224, 224])


In [13]:
import torch.nn as nn
from torchvision.models import efficientnet_b0

checkpoint = torch.load(
    BEST_MODEL_PATH,
    map_location="cpu",
    weights_only=False
)

class_names = checkpoint["class_names"]

model = efficientnet_b0(weights=None)

input_features = model.classifier[1].in_features
model.classifier = nn.Sequential(
    nn.Dropout(p=0.30),
    nn.Linear(input_features, len(class_names))
)

model.load_state_dict(checkpoint["model_state_dict"])
model = model.to(device)
model.eval()

print("Checkpoint epoch:", checkpoint["epoch"])
print(
    "Validation accuracy:",
    round(checkpoint["validation_accuracy"], 4)
)
print("Classes:", class_names)
print("Model loaded successfully.")

Checkpoint epoch: 5
Validation accuracy: 0.9463
Classes: ['CatEye', 'CatFace', 'CatMouth', 'CatNose', 'DogEye', 'DogFace', 'DogMouth', 'DogNose']
Model loaded successfully.


In [9]:
from tqdm.auto import tqdm

predicted_indices = []

model.eval()

with torch.no_grad():
    for images in tqdm(test_loader, desc="Generating test predictions"):
        images = images.to(device)

        outputs = model(images)
        predictions = outputs.argmax(dim=1)

        predicted_indices.extend(predictions.cpu().tolist())

predicted_classes = [
    class_names[index]
    for index in predicted_indices
]

print("Predictions generated:", len(predicted_classes))
print("Expected predictions:", len(sample_submission))
print("\nPredicted class distribution:")
print(pd.Series(predicted_classes).value_counts().sort_index())

Generating test predictions:   0%|          | 0/921 [00:00<?, ?it/s]

Predictions generated: 29443
Expected predictions: 29443

Predicted class distribution:
CatEye      6255
CatFace     3140
CatMouth    3345
CatNose     2887
DogEye      5414
DogFace     2871
DogMouth    2808
DogNose     2723
Name: count, dtype: int64


In [10]:
submission = sample_submission.copy()
submission["class"] = predicted_classes

allowed_classes = set(class_names)
submission_path = SUBMISSIONS_DIR / "efficientnet_b0_submission.csv"

assert len(submission) == len(sample_submission)
assert submission["image_id"].equals(sample_submission["image_id"])
assert submission["image_id"].is_unique
assert submission["class"].notna().all()
assert set(submission["class"]).issubset(allowed_classes)

submission.to_csv(submission_path, index=False)

saved_submission = pd.read_csv(
    submission_path,
    dtype={"image_id": str}
)

print("Submission saved to:", submission_path)
print("Submission shape:", saved_submission.shape)
print("Columns:", saved_submission.columns.tolist())
print("Missing values:", saved_submission.isna().sum().to_dict())
print("File size:", f"{submission_path.stat().st_size / 1024:.1f} KB")

display(saved_submission.head())
display(saved_submission.tail())

Submission saved to: /Users/amanm/Documents/KAGGLE/ENGE707-Facial-Feature-Classification/submissions/efficientnet_b0_submission.csv
Submission shape: (29443, 2)
Columns: ['image_id', 'class']
Missing values: {'image_id': 0, 'class': 0}
File size: 386.3 KB


,image_id,class
0,1,DogEye
1,2,DogMouth
2,3,DogNose
3,4,DogFace
4,5,CatMouth


,image_id,class
29438,29439,CatNose
29439,29440,CatNose
29440,29441,DogFace
29441,29442,DogFace
29442,29443,CatNose


## Kaggle Submission Result

The final EfficientNet-B0 model achieved a validation accuracy of **90.17%** and a Kaggle public leaderboard accuracy of **88.24%**.

The Kaggle result is **1.93 percentage points lower** than the validation result. This is a reasonably small difference and suggests that the validation set provided a useful estimate of performance on unseen images. The lower Kaggle score may reflect differences between the validation and hidden test distributions, as well as difficult images involving visually similar classes such as mouths and noses.

The submission contained predictions for all **29,443** test images and was accepted successfully by Kaggle.

## Submission Evidence

The submission file `efficientnet_b0_submission.csv` was successfully accepted by Kaggle and achieved a public leaderboard accuracy of **0.8824**. A screenshot of the accepted submission and score is stored in `outputs/evidence/kaggle_submission_score.png`.